In [18]:
import pandas as pd
from astropy.coordinates import SkyCoord
from astropy import units as u
import glob

# 1. 直接用 Pandas 讀取兩個 20 萬行的 CSV 檔案
df_my = sorted(glob.glob('../data/catalog/ps1/merged/*.csv'))
df_strm = sorted(glob.glob('../data/catalog/ps1/redshift_STRM/*.csv'))


In [19]:
for i in range(len(df_my)):
    df_a = pd.read_csv(df_my[i])
    df_b = pd.read_csv(df_strm[i])
    # 2. 將 RA/Dec 轉換為 Astropy 的 SkyCoord 物件
    #（假設欄位名稱為 'ra' 和 'dec'，單位是度 degree）
    cat_a = SkyCoord(ra=df_a['raMean'].values * u.deg, dec=df_a['decMean'].values * u.deg)
    cat_b = SkyCoord(ra=df_b['raMean'].values * u.deg, dec=df_b['decMean'].values * u.deg)
    # 3. 進行交叉比對（幫 cat_a 中的每一個天體，找出 cat_b 中最靠近鄰居）
    idx, d2d, _ = cat_a.match_to_catalog_sky(cat_b)

    # 4. 設定容許的極大匹配距離（例如 2 角秒）
    max_sep = 5.0 * u.arcsec
    is_match = d2d < max_sep
    # 5. 篩選並合併比對成功的資料
    results = df_a[is_match].copy()
    results['z_phot0'] = df_b.iloc[idx[is_match]]['z_phot0'].values
    results['z_photErr'] = df_b.iloc[idx[is_match]]['z_photErr'].values

    # 6. 儲存結果
    results.to_csv('../data/catalog/ps1/matched' +df_my[i].split('../data/catalog/ps1/merged')[1] , index=False)
    print(f"比對完成！共找到 {len(results)} 個匹配天體。")

比對完成！共找到 2873 個匹配天體。
比對完成！共找到 4343 個匹配天體。
比對完成！共找到 4441 個匹配天體。
比對完成！共找到 2346 個匹配天體。
比對完成！共找到 7868 個匹配天體。
比對完成！共找到 3619 個匹配天體。
比對完成！共找到 8056 個匹配天體。
比對完成！共找到 19724 個匹配天體。
比對完成！共找到 6880 個匹配天體。
比對完成！共找到 3903 個匹配天體。
比對完成！共找到 6474 個匹配天體。
比對完成！共找到 3958 個匹配天體。
比對完成！共找到 4677 個匹配天體。
比對完成！共找到 2378 個匹配天體。
比對完成！共找到 3833 個匹配天體。
比對完成！共找到 3969 個匹配天體。
比對完成！共找到 13923 個匹配天體。
比對完成！共找到 7561 個匹配天體。
比對完成！共找到 5116 個匹配天體。
比對完成！共找到 6737 個匹配天體。
比對完成！共找到 8667 個匹配天體。
比對完成！共找到 20705 個匹配天體。
比對完成！共找到 5269 個匹配天體。
比對完成！共找到 3254 個匹配天體。
比對完成！共找到 8611 個匹配天體。
比對完成！共找到 3518 個匹配天體。
比對完成！共找到 2540 個匹配天體。
比對完成！共找到 4192 個匹配天體。
比對完成！共找到 2470 個匹配天體。
比對完成！共找到 3725 個匹配天體。
比對完成！共找到 4004 個匹配天體。
比對完成！共找到 3816 個匹配天體。
比對完成！共找到 3618 個匹配天體。
比對完成！共找到 4642 個匹配天體。
比對完成！共找到 11413 個匹配天體。
比對完成！共找到 7974 個匹配天體。
比對完成！共找到 4320 個匹配天體。
比對完成！共找到 12688 個匹配天體。
比對完成！共找到 959 個匹配天體。
比對完成！共找到 988 個匹配天體。
比對完成！共找到 4015 個匹配天體。
比對完成！共找到 11354 個匹配天體。
比對完成！共找到 3509 個匹配天體。
比對完成！共找到 4609 個匹配天體。
比對完成！共找到 3085 個匹配天體。
比對完成！共找到 6267 個匹配天體。
比對完成！共找到 3750 個匹配天體。
比對完成！共找到 

In [20]:
len(glob.glob('../data/catalog/ps1/matched/*.csv'))

492

In [26]:
import pandas as pd
from astropy.table import Table
import astropy.units as u
from astroquery.xmatch import XMatch
import glob

# 1. 讀取您的 20 萬行本地 CSV 並轉換為 Astropy Table
df_local = glob.glob('../data/catalog/ps1/matched/*.csv')

i = 0  # 選擇要比對的 CSV 檔案索引
df_a = pd.read_csv(df_local[i])
local_table = Table.from_pandas(df_a)

# 2. 設定匹配參數（以 SDSS DR12 為例）
# 如果要使用其他版本的 SDSS，請替換 cat2 的 VizieR ID
sdss_dr12_id = 'vizier:V/147/sdss12' 

print("正在與 SDSS 進行雲端交叉比對...")

# 3. 執行跨目錄比對（設定容許距離為 2 角秒）
matched_table = XMatch.query(
    cat1=local_table,
    cat2=sdss_dr12_id,
    max_distance=0.2 * u.arcsec,
    colRA1='raMean',      # 您本地 CSV 的 RA 欄位名稱
    colDec1='decMean'     # 您本地 CSV 的 Dec 欄位名稱
)

# 4. 將結果轉回 Pandas DataFrame 並儲存
df_results = matched_table.to_pandas()
df_results.to_csv('../data/catalog/ps1/matched/sdss' +df_local[i].split('../data/catalog/ps1/matched')[1] , index=False)

print(f"比對完成！共找到 {len(df_results)} 個 SDSS 匹配天體。")


正在與 SDSS 進行雲端交叉比對...
比對完成！共找到 2690 個 SDSS 匹配天體。
